# PanoraDent AI — entraînement sur Colab

**Le code vit dans le dépôt, seul le calcul tourne ici.** Ce notebook ne contient aucune logique : il clone `PanoraDent-AI` et appelle les scripts de `training/`. Toute modification d'algorithme se fait dans le dépôt, pas dans ce fichier — sans quoi l'entraînement cesse d'être reproductible.

**Avant de lancer** : `Exécution → Modifier le type d'exécution → GPU (T4)`.

⚠️ Une session Colab est éphémère. Les données doivent venir de Google Drive, et les poids doivent en repartir avant la déconnexion.

## 1. Vérifier le GPU

Sans GPU, l'entraînement est inexploitable — mieux vaut le savoir tout de suite.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import torch; print('CUDA disponible :', torch.cuda.is_available())

## 2. Dépôt et dépendances

In [ ]:
%cd /content
!git clone --depth 1 https://github.com/STAGE-M2-ORION/PanoraDent-AI.git panoradent
%cd /content/panoradent
!pip install -q ultralytics==8.2.0
!git rev-parse --short HEAD

## 3. Données depuis Drive

Déposer sur Drive l'archive du dataset (annotations COCO + images). **Ne jamais committer de données dans le dépôt** : `.gitignore` bloque déjà `data/` et `*.dcm`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Adapter au chemin réel sur le Drive.
ARCHIVE = '/content/drive/MyDrive/panoradent/dentex.zip'
!mkdir -p data && unzip -q -o "$ARCHIVE" -d data/
!ls -la data/

## 4. Préparer le dataset

Découpage **par patient**, jeu de test gelé, graine fixe. Lire attentivement la sortie : le tableau de distribution révèle le déséquilibre des classes, et toute catégorie non mappée est signalée.

In [ ]:
!python training/prepare_dataset.py \
    --annotations data/dentex/annotations.json \
    --images      data/dentex/images \
    --out         data/yolo \
    --class-map   training/class_map.dentex.json \
    --seed 42

## 5. Entraîner

`yolo11n` d'abord : rapide, il valide la chaîne et donne une référence basse. Passer à `yolo11s`/`yolo11m` seulement une fois qu'un premier chiffre existe.

`imgsz` doit rester identique à `MODEL_IMAGE_SIZE` du service d'inférence.

In [ ]:
!python training/train.py \
    --data data/yolo/data.yaml \
    --model yolo11n.pt \
    --epochs 100 --imgsz 640 --batch 16 \
    --device 0 --seed 42 \
    --name colab-run

## 6. Évaluer sur le test gelé

**Une seule fois.** Régler les hyperparamètres en regardant le test, c'est le contaminer : les chiffres cessent d'être une mesure indépendante. Pour itérer, utiliser `--split val`.

In [ ]:
!python training/evaluate.py \
    --weights runs/colab-run/weights/best.pt \
    --data    data/yolo/data.yaml \
    --split   test --conf 0.25

## 7. Récupérer les résultats

Vers Drive — la session sera perdue. Puis, **sur la machine qui héberge la stack** :

```bash
python training/publish_model.py \
    --version yolov11n-dentex-2026.09 \
    --dataset "DENTEX, split patient 70/15/15, graine 42" \
    --metrics metrics.json --weights best.pt --install
```

In [ ]:
import shutil, pathlib

SORTIE = pathlib.Path('/content/drive/MyDrive/panoradent/modeles/colab-run')
SORTIE.mkdir(parents=True, exist_ok=True)
for fichier in ('weights/best.pt', 'weights/metrics.json', 'run.json',
                'results.csv', 'confusion_matrix_normalized.png'):
    source = pathlib.Path('runs/colab-run') / fichier
    if source.exists():
        shutil.copy2(source, SORTIE / source.name)
        print('copié :', source.name)
    else:
        print('absent :', fichier)